# Content Factory - render assistido no Colab
[Open in Colab](https://colab.research.google.com/github/mrJoao28/Content-Factory/blob/b294cd057fc826ee5eb02044b9ab3bbd196985ab/notebooks/content_factory_colab.ipynb)

Use **ambiente hospedado**, Python 3. Em Ambiente de execução - Alterar tipo de ambiente de execução, escolha GPU disponível. Runtime local usa seu PC e não resolve o objetivo.
Execute as células manualmente, em ordem. Este notebook não hospeda servidor, não abre túneis e não chama ChatGPT. O pacote contém storyboard e áudio privados: compartilhe somente se desejar. Nunca envie .env, tokens, banco ou credenciais. Os resultados precisam ser importados para revisão humana.

GPU, duração e disponibilidade não são garantidas. A sessão gratuita pode terminar antes do teto de 12 horas; não há garantia de 60 horas em três dias. Não use keep-alive, contas múltiplas, reconexão automática ou workers distribuídos. Comprar espaço no Drive não aumenta cota de GPU. Leia a [FAQ oficial](https://research.google.com/colaboratory/faq.html).

In [ ]:
import os  # noqa: E402
import platform  # noqa: E402
import shutil  # noqa: E402
import subprocess  # noqa: E402
import sys  # noqa: E402
from pathlib import Path  # noqa: E402

WORK = Path("/content/content-factory-work")
WORK.mkdir(parents=True, exist_ok=True)
print(
    {
        "python": platform.python_version(),
        "system": platform.system(),
        "free_disk_GiB": round(shutil.disk_usage(WORK).free / 1024**3, 2),
    }
)
if platform.system() != "Linux":
    raise RuntimeError("Selecione um ambiente hospedado Linux do Colab, não runtime local.")
print({"available_RAM_GiB": round(os.sysconf("SC_AVPHYS_PAGES") * os.sysconf("SC_PAGE_SIZE") / 1024**3, 2)})
if shutil.which("nvidia-smi"):
    print(
        subprocess.run(
            ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
            capture_output=True,
            text=True,
            timeout=20,
        ).stdout
    )
else:
    print("nvidia-smi indisponível. GPU não comprovada. CPU exige escolha explícita na exportação.")

## Instalar ferramentas e carregar código confiável
Versão Linux fixada, checksum oficial embutido. Não altera drivers/CUDA. Somente Pydantic/Pillow para o renderer; sem servidor, OAuth ou vault. O commit abaixo foi escolhido no notebook, nunca pelo ZIP. Se o pacote pedir outra revisão, revise e selecione explicitamente o commit confiável antes de recriar o ambiente.

In [ ]:
import hashlib  # noqa: E402
import tarfile  # noqa: E402
import urllib.request  # noqa: E402

RENDER_COMMIT = "fed2d65b275ef984cdd84c6fe09076aea53d4a70"
REPOSITORY = "https://github.com/mrJoao28/Content-Factory"
BLENDER_VERSION = "4.5.3"
BLENDER_SHA256 = "975c58fcb244273838534bba771e64ad87739216b0f9b39a888531a49a72d845"


def download(url, destination):
    if not destination.exists():
        urllib.request.urlretrieve(url, destination)


def checksum(path):
    h = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024**2), b""):
            h.update(block)
    return h.hexdigest()


archive = WORK / f"blender-{BLENDER_VERSION}-linux-x64.tar.xz"
download(f"https://download.blender.org/release/Blender4.5/{archive.name}", archive)
if checksum(archive) != BLENDER_SHA256:
    raise RuntimeError("Checksum Blender divergente. Remova apenas este download incompleto e repita.")
BLENDER = WORK / f"blender-{BLENDER_VERSION}-linux-x64/blender"
if not BLENDER.exists():
    with tarfile.open(archive) as source:
        source.extractall(WORK, filter="data")
subprocess.run(["apt-get", "update", "-qq"], check=True, timeout=300)
subprocess.run(
    [
        "apt-get",
        "install",
        "-y",
        "-qq",
        "ffmpeg",
        "libxi6",
        "libxrender1",
        "libxxf86vm1",
        "libgl1",
        "libegl1",
    ],
    check=True,
    timeout=300,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "pydantic==2.12.5", "pillow==12.1.1"],
    check=True,
    timeout=300,
)
code_archive = WORK / f"{RENDER_COMMIT}.tar.gz"
download(f"{REPOSITORY}/archive/{RENDER_COMMIT}.tar.gz", code_archive)
REPO = WORK / f"Content-Factory-{RENDER_COMMIT}"
if not REPO.exists():
    with tarfile.open(code_archive) as source:
        source.extractall(WORK, filter="data")
sys.path.insert(0, str(REPO / "services/api"))
from content_factory_render.archive import MAX_ZIP  # noqa: E402
from content_factory_render.runner import Renderer, load_package  # noqa: E402

TEMPLATE = REPO / "templates/blender/factory_scene.py"
print(
    subprocess.run(
        [str(BLENDER), "--version"], capture_output=True, text=True, check=True, timeout=30
    ).stdout.splitlines()[0]
)

## Drive opcional - autorize somente ao executar esta célula
Crie/seleciona a pasta. A autorização padrão do Colab funciona independentemente da integração Drive da aplicação Windows. Sem Drive, /content permanece tempor?rio e arquivos podem desaparecer. Não são necessários Google Cloud Console, client secret ou conta de serviço.

In [ ]:
MOUNT_DRIVE = False  # @param {type:"boolean"}
DRIVE_FOLDER = "/content/drive/MyDrive/ContentFactory"  # @param {type:"string"}
PERSISTENT = None
if MOUNT_DRIVE:
    from google.colab import drive  # noqa: E402

    drive.mount("/content/drive")
    PERSISTENT = Path(DRIVE_FOLDER)
    PERSISTENT.mkdir(parents=True, exist_ok=True)
else:
    print("Sem persistência: faça download manual antes de sair. Nenhum checkpoint está salvo no Drive.")

In [ ]:
INPUT_MODE = "upload"  # @param ["upload", "drive"]
DRIVE_ZIP = "/content/drive/MyDrive/ContentFactory/input.zip"  # @param {type:"string"}
import tempfile  # noqa: E402

if INPUT_MODE == "upload":
    from google.colab import files  # noqa: E402

    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Selecione somente um ZIP de até 256 MiB.")
    payload = next(iter(uploaded.values()))
    if len(payload) > MAX_ZIP:
        raise ValueError("ZIP excede 256 MiB.")
    INPUT = WORK / "input.zip"
    INPUT.write_bytes(payload)
    del payload, uploaded
else:
    if not MOUNT_DRIVE:
        raise ValueError("Autorize o Drive na célula anterior.")
    INPUT = Path(DRIVE_ZIP)
EXTRACTED = Path(tempfile.mkdtemp(prefix="package-", dir=WORK))
package = load_package(INPUT, EXTRACTED, trusted_commit=RENDER_COMMIT, template=TEMPLATE)
print(
    {
        "project_id": str(package.project_id),
        "revision": package.revision,
        "export_id": str(package.export_id),
        "seconds": package.board.duration,
        "frames": round(package.board.duration * package.profile.fps),
        "profile": package.profile.model_dump(),
    }
)
if PERSISTENT:
    shutil.copyfile(INPUT, PERSISTENT / f"{package.export_id}-input.zip")

## Controles e teste real curto
Qualidade/resolução/FPS/samples/seed seguem o pacote registrado localmente. Para alterá-los, volte - fábrica e exporte o perfil desejado; retornos divergentes são rejeitados. A política GPU permite testar OptiX e depois CUDA; CPU somente pode ser aceita quando exportada explicitamente. O teste realmente renderiza 12-24 frames da cena recebida. Listar GPU não prova renderização. Observe as imagens antes de continuar.

In [ ]:
PROBE_FRAMES = 12  # @param {type:"slider",min:12,max:24,step:1}
THREADS = 2  # @param {type:"slider",min:1,max:8,step:1}
TIME_LIMIT_SECONDS = 1800  # @param {type:"integer"}
DISK_BUDGET_MB = 8192  # @param {type:"integer"}
BATCH_FRAMES = 24  # @param {type:"slider",min:1,max:120,step:1}
if not 30 <= TIME_LIMIT_SECONDS <= 21600:
    raise ValueError("Limite deve ser finito, entre 30 e 21600 segundos.")
policy = package.profile.hardware
backends = ["OPTIX", "CUDA"] if policy == "GPU" else [policy]
renderer = None
from IPython.display import Image, display  # noqa: E402

for backend in backends:
    print("Testando Cycles", backend, "em subprocesso; sem fallback CPU.")
    try:
        candidate = Renderer(
            package,
            EXTRACTED,
            WORK / "frames",
            blender=BLENDER,
            template=TEMPLATE,
            threads=THREADS,
            backend=backend,
            timeout=TIME_LIMIT_SECONDS,
            disk_mb=DISK_BUDGET_MB,
        )
        if PERSISTENT:
            print("Frames restaurados:", candidate.restore(PERSISTENT / "checkpoints"))
        candidate.timeout = min(TIME_LIMIT_SECONDS, 300) if backend != "CPU" else TIME_LIMIT_SECONDS
        measurement = candidate.batch(1, min(PROBE_FRAMES, candidate.total))
        candidate.timeout = TIME_LIMIT_SECONDS
        renderer = candidate
        print(measurement)
        for n in sorted(set([1, min(PROBE_FRAMES, renderer.total) // 2, min(PROBE_FRAMES, renderer.total)])):
            display(Image(filename=str(renderer.folder / f"{max(1, n):06}.png")))
        if measurement["seconds_per_frame"]:
            print(
                "Estimativa restante em minutos (varia entre cenas):",
                round(measurement["seconds_per_frame"] * (renderer.total - measurement["completed"]) / 60, 1),
            )
        break
    except (RuntimeError, ValueError, TimeoutError) as error:
        print(backend, "não passou no teste:", str(error))
        if policy == "GPU" and backend == "OPTIX":
            print("Tentando CUDA, conforme política visível.")
if renderer is None:
    raise RuntimeError(
        "Nenhum backend autorizado passou. Verifique runtime/VRAM/disco ou reexporte explicitamente com CPU. Render longo não iniciado."
    )

## Render completo - execute somente depois de inspecionar o teste
Progresso real a cada lote; uma execução finita. Interromper a célula encerra o subprocesso. Com Drive, cada lote gera shard compactado e índice publicado após verificar sua cópia. Não grava PNGs individualmente no Drive. Uma interrupção pode perder o lote desde o último índice salvo. Não use reconexão automática ou keep-alive.

In [ ]:
if renderer is None:
    raise RuntimeError("Execute o teste curto primeiro.")
renderer.render(persistent=PERSISTENT / "checkpoints" if PERSISTENT else None, batch_size=BATCH_FRAMES)
print("Todos os frames desta revisão/configuração foram validados.")

## Retomar em outra sessão
Reexecute instalação, Drive e carregamento do **mesmo ZIP**, com mesmo Blender/backend/perfil/dispositivo compatével. A célula do teste restaura shards antes de renderizar frames ausentes. Para tentar restauração novamente nesta sessão, execute abaixo e depois a célula de render completo. Shards sem índice ou de outro fingerprint não são reutilizados. Os 100 frames locais Windows/CPU não são importados automaticamente.

In [ ]:
if renderer is None or PERSISTENT is None:
    print("Retomada persistente requer renderer testado e Drive autorizado.")
else:
    print("Frames válidos restaurados:", renderer.restore(PERSISTENT / "checkpoints"))

## Montar MP4, validar, assistir e baixar
O FFmpeg decodifica o vídeo; a aplicação fará sua própria validação completa na importação. Nenhum resultado será aprovado/publicado aqui. Não - necessário transportar milhares de PNGs.

In [ ]:
RESULT = WORK / f"{package.export_id}-result.zip"
renderer.finish(RESULT)
from IPython.display import Video  # noqa: E402

if (renderer.folder / "video.mp4").stat().st_size <= 64 * 1024**2:
    display(Video(filename=str(renderer.folder / "video.mp4"), embed=True))
else:
    print("Vídeo grande: baixe e assista localmente antes da revisão.")
if PERSISTENT:
    partial = PERSISTENT / f"{RESULT.name}.partial"
    shutil.copyfile(RESULT, partial)
    if checksum(partial) != checksum(RESULT):
        raise ValueError("Cópia de retorno falhou na integridade.")
    partial.replace(PERSISTENT / RESULT.name)
from google.colab import files  # noqa: E402

files.download(str(RESULT))
print(
    "Na fábrica Windows: abra a produção original - Import result ZIP - assistir - aprovar o hash atual explicitamente."
)